# Avance 4. Modelos alternativos

## Equipo 41

**Proyecto:** Evaluación de modelos de visión computacional para la detección de eventos de ocultamiento de mercancía en video

**Integrantes:**
- Jesús Alberto Jiménez Ramos
- Nélida Silva Cabrera
- José Luis Tinajero Guerrero

**Empresa:** BLOCK NETWORKS S.A. DE C.V. (BNext)

**Asesor:** Dr. Guillermo Mota Medina

**Curso:** Proyecto Integrador  
**Maestría en Inteligencia Artificial Aplicada**  
**Tecnológico de Monterrey**

**Fecha:** octubre de 2026

## 1. Objetivo del avance

El objetivo de este notebook es evaluar modelos alternativos para la detección de eventos de ocultamiento de mercancía y determinar cuáles presentan el comportamiento más prometedor para continuar con el proyecto.

El avance parte del modelo baseline R3D-18 desarrollado en el Avance 3 y amplía la experimentación mediante arquitecturas con diferentes mecanismos de modelado espacio-temporal. Se evaluarán seis modelos individuales:

1. R3D-18.
2. MC3-18.
3. R(2+1)D-18.
4. S3D.
5. MViT-V2-S.
6. STG-NF sobre secuencias de pose.

Los primeros cinco modelos procesan información RGB y se evaluarán bajo un protocolo común de clasificación binaria. STG-NF utiliza una representación diferente, basada en puntos corporales, y formula el problema como detección de anomalías. Por esta razón, sus resultados se reportarán como un experimento complementario dentro de la comparación de modelos, evitando interpretar sus métricas como directamente equivalentes a las obtenidas por los clasificadores RGB cuando los datos y las unidades de evaluación sean diferentes.

Además de comparar los modelos, se seleccionarán los dos candidatos con mayor evidencia de desempeño y generalización dentro de condiciones experimentales comparables. Sobre estos candidatos se realizará ajuste de hiperparámetros antes de seleccionar el modelo individual que continuará hacia la siguiente etapa del proyecto.

La evaluación considera no solamente el desempeño predictivo, sino también estabilidad, errores observados, tiempo de entrenamiento, demanda computacional y pertinencia para el caso de uso planteado por BNext.

## 2. Relación entre la rúbrica y el diseño experimental

La rúbrica del Avance 4 solicita construir al menos seis modelos individuales utilizando algoritmos variados, comparar su rendimiento, seleccionar los dos modelos más prometedores, ajustar sus hiperparámetros y elegir un modelo individual final.

Debido a que este proyecto trabaja con reconocimiento de acciones en video, los criterios de la rúbrica se adaptan de la siguiente manera:

| Criterio de la rúbrica | Aplicación en este notebook |
|---|---|
| Seis modelos diferentes | Se evalúan cinco arquitecturas para video RGB y un modelo basado en secuencias de pose. Los modelos representan diferentes mecanismos para capturar información espacial y temporal. |
| Comparación de desempeño | Los modelos RGB se comparan bajo las mismas particiones y definición de clases. Average Precision (AP) se utiliza como métrica principal. Se reportan además ROC-AUC, PR-AUC, EER, precision, recall, F1-score y tiempos de entrenamiento. |
| Modelos con diferente representación | STG-NF utiliza secuencias corporales de pose y detección de anomalías. Se reporta dentro de los seis modelos evaluados, pero su comparación con RGB se interpreta considerando las diferencias de representación, datos y objetivo de aprendizaje. |
| Selección de dos modelos | La selección se realiza utilizando exclusivamente resultados de validación y considerando la métrica principal, métricas complementarias, estabilidad y costo computacional. El conjunto de prueba no se utiliza para seleccionar candidatos ni hiperparámetros. |
| Ajuste fino | Los dos modelos seleccionados se someten a experimentos de ajuste de hiperparámetros utilizando únicamente entrenamiento y validación. |
| Modelo individual final | El modelo final se selecciona considerando desempeño, errores, capacidad de generalización, complejidad y requerimientos computacionales, además de la métrica principal. |

La comparación pretende identificar diferencias reales entre enfoques y evitar atribuir mejoras a cambios en datos, particiones o procedimientos de evaluación.

## 3. Punto de partida: modelo baseline

El Avance 3 estableció R3D-18 como modelo baseline para el problema de clasificación binaria de clips de video. El modelo utiliza pesos preentrenados en Kinetics-400 y fue ajustado para distinguir las clases `ocultamiento` y `sin_ocultamiento`.

La unidad de análisis utilizada para los modelos RGB es un clip compuesto por 16 frames con resolución espacial de 112 × 112 píxeles:

$$
X_i \in \mathbb{R}^{3 \times 16 \times 112 \times 112}
$$

La variable objetivo se define como:

$$
y_i =
\begin{cases}
0, & \text{sin\_ocultamiento} \\
1, & \text{ocultamiento}
\end{cases}
$$


El Avance 3 también permitió identificar casos donde las etiquetas originales asociadas con `shoplifting` no correspondían con suficiente claridad a la definición operacional de ocultamiento utilizada por el proyecto. Estos casos se analizaron cualitativamente y constituyen el punto de partida para una revisión dirigida del ground truth antes de realizar la comparación de modelos del presente avance.

## 4. Experimento 0: calidad del ground truth

Antes de evaluar modelos alternativos se realiza un experimento dirigido a analizar el efecto de la calidad de las etiquetas sobre las métricas del baseline.

Durante el análisis de errores del Avance 3 se revisaron los 13 falsos negativos obtenidos por R3D-18 sobre el conjunto de prueba. Once de ellos fueron clasificados como ambiguos respecto a la definición operacional de `ocultamiento`, mientras que dos presentaron evidencia suficiente para conservar la etiqueta positiva.

La revisión no constituye una nueva auditoría completa del dataset. Su alcance se limita a los casos que ya fueron identificados y revisados durante el análisis cualitativo del Avance 3.

Los clips ambiguos:

- permanecen documentados para conservar trazabilidad;
- no se reasignan automáticamente a `sin_ocultamiento`;
- se excluyen del conjunto utilizado para calcular las métricas binarias del Avance 4.

Los clips cuya etiqueta fue considerada clara conservan su clase original.

Las particiones de entrenamiento, validación y prueba no se redistribuyen después de esta revisión. Por lo tanto, los grupos y videos mantienen la partición asignada previamente.

El conjunto resultante se denomina en este notebook **dataset v2**. Una vez aplicada la revisión dirigida, sus etiquetas y particiones quedan congeladas para todos los experimentos posteriores.

El Experimento 0 compara las métricas del mismo checkpoint R3D-18 utilizando:

1. El ground truth empleado en el Avance 3.
2. El ground truth revisado utilizado en el Avance 4.

Cualquier diferencia entre ambas evaluaciones representa un cambio en la estimación del desempeño debido a la calidad y alineación semántica del ground truth. No representa una mejora del modelo, ya que los parámetros y predicciones de R3D-18 permanecen sin cambios.

## 5. Definiciones y decisiones de modelado

### 5.1 Definición operacional de la clase positiva

Se considera `ocultamiento` cuando el clip contiene evidencia visual suficiente de que una persona aproxima o introduce mercancía hacia una zona de ocultamiento definida para el proyecto, como ropa, bolsillo, bolsa o mochila.

La clase `sin_ocultamiento` incluye situaciones en las que la mercancía permanece visible o donde no ocurre la acción objetivo.

Los clips cuya acción no puede determinarse con suficiente claridad se consideran ambiguos y quedan fuera de la evaluación binaria principal.

### 5.2 Unidad de análisis para modelos RGB

Los modelos RGB procesan clips temporales en lugar de videos completos. Esta decisión permite asociar la predicción con el segmento donde puede observarse la acción objetivo y evita transferir automáticamente la etiqueta general de un video a todos sus fragmentos.

La configuración inicial utiliza 16 frames por clip. Las transformaciones espaciales requeridas por cada arquitectura se documentarán cuando difieran del preprocesamiento utilizado por R3D-18.

### 5.3 Unidad de análisis para poses

STG-NF utiliza ventanas temporales de puntos corporales correspondientes a una persona. La representación de RetailS utiliza puntos corporales en formato COCO17 e incluye coordenadas y valores de confianza.

Este enfoque modela principalmente patrones de movimiento corporal y no dispone directamente de información visual detallada sobre el producto, textura, vestimenta o destino final de la mercancía.

### 5.4 Tipo de aprendizaje

Los modelos RGB se formulan como clasificadores supervisados binarios.

STG-NF se utiliza como modelo de detección de anomalías sobre secuencias de pose. Aprende patrones asociados con actividad normal y genera una puntuación de anomalía para las secuencias evaluadas.

La puntuación de STG-NF no se interpreta directamente como una probabilidad de ocultamiento.

### 5.5 Transferencia de aprendizaje

Cuando se disponga de pesos preentrenados compatibles, se utilizarán como punto de partida para reducir el costo de entrenamiento y aprovechar representaciones aprendidas sobre conjuntos de video de mayor escala.

Las fuentes de los pesos, transformaciones requeridas y capas modificadas para cada arquitectura se documentarán en la sección correspondiente a cada experimento.

## 6. Criterios de calidad experimental

Los experimentos se diseñan para que las diferencias observadas entre modelos puedan atribuirse principalmente al enfoque de modelado y no a cambios inadvertidos en los datos o en el procedimiento de evaluación.

### 6.1 Calidad de las etiquetas

Las etiquetas utilizadas deben corresponder con la definición operacional de `ocultamiento` adoptada por el proyecto. Los casos ambiguos identificados durante la revisión del Avance 3 se excluyen de la evaluación binaria del dataset v2.

### 6.2 Independencia entre particiones

Todos los clips derivados de un mismo video o grupo relacionado permanecen dentro de una sola partición. No se redistribuyen ejemplos después de observar el desempeño de los modelos.

### 6.3 Separación entre selección y evaluación final

El conjunto de entrenamiento se utiliza para optimizar los parámetros del modelo.

El conjunto de validación se utiliza para:

- seleccionar checkpoints;
- comparar configuraciones;
- seleccionar los dos modelos candidatos;
- ajustar hiperparámetros;
- definir el umbral de decisión cuando corresponda.

El conjunto de prueba se reserva para estimar el desempeño de configuraciones ya seleccionadas.

### 6.4 Reproducibilidad

Cada experimento deberá registrar, cuando aplique:

- Arquitectura.
- Inicialización y pesos preentrenados.
- Semilla aleatoria.
- Tamaño de lote.
- Optimizador.
- Tasa de aprendizaje.
- Regularización.
- Número máximo de épocas.
- Criterio de early stopping.
- Checkpoint seleccionado.
- Tiempo de entrenamiento.
- Dispositivo utilizado.

### 6.5 Comparabilidad

Los cinco modelos RGB utilizan las mismas particiones y la misma definición de las clases.

Cuando una arquitectura requiera una resolución, normalización u otra transformación específica, la diferencia se documentará explícitamente y no se interpretará como un cambio en el ground truth.

STG-NF se analiza mediante un protocolo separado porque utiliza otra representación y otro conjunto de datos.

### 6.6 Control del conjunto de prueba

Después de congelar el dataset v2, las etiquetas de prueba no se modificarán en función de los errores producidos por los modelos del Avance 4.

Los errores posteriores podrán analizarse cualitativamente, pero dicho análisis no se utilizará para alterar el ground truth de esta ronda experimental.

## 7. Métricas de evaluación

Debido al desbalance entre `ocultamiento` y `sin_ocultamiento`, la evaluación no se basa únicamente en accuracy.

### 7.1 Métrica principal

**Average Precision (AP)** se utiliza como métrica principal para la selección y comparación de los modelos RGB.

AP resume el comportamiento precision-recall a través de diferentes umbrales y mantiene continuidad con el criterio utilizado para seleccionar el checkpoint del baseline en el Avance 3.

### 7.2 Métricas complementarias

Se reportarán también:

- PR-AUC.
- ROC-AUC.
- Equal Error Rate (EER).
- Precision.
- Recall.
- F1-score.
- Specificity.
- Accuracy.
- Matriz de confusión.

Las métricas dependientes de un umbral se calcularán utilizando un umbral seleccionado exclusivamente a partir del conjunto de validación.

### 7.3 Métricas computacionales

La rúbrica solicita registrar tiempos de entrenamiento. Adicionalmente, cuando sea posible, se documentarán:

- Tiempo total de entrenamiento.
- Tiempo medio por época.
- Throughput o tiempo de inferencia.
- Memoria GPU observada.
- Número aproximado de parámetros.

Estas mediciones permitirán complementar la selección del modelo con información sobre costo computacional y posible viabilidad para una implementación futura.

## 8. Protocolos experimentales

El Avance 4 se divide en tres bloques experimentales.

### 8.1 Experimento 0: sensibilidad de las métricas a la calidad del ground truth

Se reutiliza el checkpoint R3D-18 del Avance 3 y se comparan sus métricas utilizando el ground truth original y el ground truth revisado.

No se realiza entrenamiento adicional.

Este experimento permite cuantificar cuánto cambia la evaluación del baseline cuando los casos ambiguos previamente identificados se excluyen de la tarea binaria.

### 8.2 Experimentos RGB

Se evaluarán cinco arquitecturas:

| ID | Arquitectura | Familia |
|---|---|---|
| RGB-0 | R3D-18 | Convolución 3D |
| RGB-1 | MC3-18 | Convolución 3D/2D mixta |
| RGB-2 | R(2+1)D-18 | Convolución espacial y temporal factorizada |
| RGB-3 | S3D | Convolución 3D separable |
| RGB-4 | MViT-V2-S | Transformer multiescala para video |

R3D-18 funciona como baseline.

Los otros cuatro modelos constituyen alternativas con diferentes mecanismos de procesamiento espacio-temporal.

Todos se compararán utilizando las mismas particiones del dataset v2. La selección inicial de candidatos se realizará con el conjunto de validación y utilizando AP como métrica principal.

### 8.3 Experimento de poses

El sexto modelo es STG-NF.

Se utilizarán las particiones disponibles de RetailS y su representación basada en pose COCO17.

El entrenamiento se realizará utilizando actividad normal conforme al planteamiento del modelo de detección de anomalías. Los conjuntos de prueba `RetailS_test_staged` y `RetailS_test_realworld` se reportarán por separado.

Cuando se requiera seleccionar un umbral y no exista un conjunto de validación con anomalías reservado para este propósito, la decisión se realizará utilizando exclusivamente datos de validación derivados del conjunto de entrenamiento normal y un criterio explícito de tasa de falsas alarmas.

Los resultados de STG-NF se incorporarán a la síntesis de los seis modelos, pero se evitará establecer un ranking numérico directo contra los clasificadores RGB cuando las unidades de evaluación o las etiquetas no sean equivalentes.

### 8.4 Selección de candidatos para ajuste fino

Después de la evaluación inicial se seleccionarán dos modelos para optimización de hiperparámetros.

La selección se sustentará en:

1. AP de validación.
2. Métricas complementarias.
3. Brecha entre entrenamiento y validación.
4. Análisis de falsos positivos y falsos negativos.
5. Tiempo y recursos de entrenamiento.
6. Compatibilidad con el objetivo técnico del proyecto.

El conjunto de prueba no participará en esta decisión.

In [1]:
# ============================================================
# Configuración general del Avance 4
# ============================================================

SEED = 42

PRIMARY_METRIC = "average_precision"

RGB_MODELS = {
    "RGB-0": "R3D-18",
    "RGB-1": "MC3-18",
    "RGB-2": "R(2+1)D-18",
    "RGB-3": "S3D",
    "RGB-4": "MViT-V2-S",
}

POSE_MODEL = {
    "POSE-0": "STG-NF"
}

AMBIGUOUS_TEST_CLIPS_AVANCE3 = [
    # DCSASS
    "Shoplifting006_x264_28",
    "Shoplifting006_x264_29",
    "Shoplifting006_x264_19",
    "Shoplifting021_x264_23",

    # MNNIT
    "MNNIT_065",
    "MNNIT_067",
    "MNNIT_013",
    "MNNIT_064",
    "MNNIT_069",
    "MNNIT_066",
    "MNNIT_068",
]

print("=== CONFIGURACIÓN DEL AVANCE 4 ===")
print(f"Semilla: {SEED}")
print(f"Métrica principal: {PRIMARY_METRIC}")
print(f"Modelos RGB: {len(RGB_MODELS)}")
print(f"Modelo de poses: {list(POSE_MODEL.values())[0]}")
print(f"Total de modelos: {len(RGB_MODELS) + len(POSE_MODEL)}")
print(
    "Clips ambiguos identificados en el Avance 3:",
    len(AMBIGUOUS_TEST_CLIPS_AVANCE3)
)

=== CONFIGURACIÓN DEL AVANCE 4 ===
Semilla: 42
Métrica principal: average_precision
Modelos RGB: 5
Modelo de poses: STG-NF
Total de modelos: 6
Clips ambiguos identificados en el Avance 3: 11


In [3]:
from pathlib import Path
import os
import pandas as pd


def find_repo_root():
    """
    Localiza la raíz del repositorio sin depender de rutas
    personales específicas del equipo.
    """

    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
    ]

    repo_env = os.environ.get("MNA_REPO_DIR")

    if repo_env:
        candidates.append(Path(repo_env))

    candidates.append(
        Path("/opt/workspace/MNA---Proyecto-Integrador")
    )

    for candidate in candidates:
        data_dir = (
            candidate
            / "Datos"
            / "Preparacion_Datasets"
        )

        if data_dir.is_dir():
            return candidate

    if not TEST_PREDICTIONS_PATH.exists():
        raise FileNotFoundError(
            "No se encontró el archivo esperado: "
            "Notebooks/checkpoints/"
            "r3d18_baseline_test_predictions.csv"
        )


REPO_DIR = find_repo_root()

CHECKPOINT_DIR = (
    REPO_DIR
    / "Notebooks"
    / "checkpoints"
)

TEST_PREDICTIONS_PATH = (
    CHECKPOINT_DIR
    / "r3d18_baseline_test_predictions.csv"
)

if not TEST_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {TEST_PREDICTIONS_PATH}"
    )

df_test_v1 = pd.read_csv(
    TEST_PREDICTIONS_PATH
)

print("=== PREDICCIONES BASELINE CARGADAS ===")
print("Archivo:", TEST_PREDICTIONS_PATH.relative_to(REPO_DIR))
print("Registros:", len(df_test_v1))
print(
    "Positivos reales:",
    int(df_test_v1["clase_binaria"].sum())
)
print(
    "Negativos reales:",
    int(
        len(df_test_v1)
        - df_test_v1["clase_binaria"].sum()
    )
)
print(
    "Split:",
    df_test_v1["split"].unique().tolist()
)

=== PREDICCIONES BASELINE CARGADAS ===
Archivo: Notebooks/checkpoints/r3d18_baseline_test_predictions.csv
Registros: 147
Positivos reales: 28
Negativos reales: 119
Split: ['test']


In [4]:
# ============================================================
# Validación de clips ambiguos identificados en el Avance 3
# ============================================================

df_ambiguos_v1 = (
    df_test_v1[
        df_test_v1["clip_id"].isin(
            AMBIGUOUS_TEST_CLIPS_AVANCE3
        )
    ]
    .copy()
    .sort_values(
        ["dataset", "video_id", "clip_id"]
    )
    .reset_index(drop=True)
)

clips_encontrados = set(
    df_ambiguos_v1["clip_id"]
)

clips_esperados = set(
    AMBIGUOUS_TEST_CLIPS_AVANCE3
)

clips_faltantes = (
    clips_esperados
    - clips_encontrados
)

duplicados = (
    df_ambiguos_v1["clip_id"]
    .duplicated()
    .sum()
)

todos_positivos = (
    df_ambiguos_v1["clase_binaria"]
    .eq(1)
    .all()
)

print("=== VALIDACIÓN DE CLIPS AMBIGUOS ===")
print(
    "Clips ambiguos esperados:",
    len(clips_esperados)
)
print(
    "Clips ambiguos encontrados:",
    len(df_ambiguos_v1)
)
print(
    "Clips faltantes:",
    len(clips_faltantes)
)
print(
    "Registros duplicados:",
    int(duplicados)
)
print(
    "Todos tienen etiqueta positiva:",
    todos_positivos
)

assert len(df_ambiguos_v1) == 11
assert len(clips_faltantes) == 0
assert duplicados == 0
assert todos_positivos

display(
    df_ambiguos_v1[
        [
            "dataset",
            "video_id",
            "clip_id",
            "clase_binaria",
            "p_ocultamiento",
            "pred_threshold_final"
        ]
    ]
)

=== VALIDACIÓN DE CLIPS AMBIGUOS ===
Clips ambiguos esperados: 11
Clips ambiguos encontrados: 11
Clips faltantes: 0
Registros duplicados: 0
Todos tienen etiqueta positiva: True


,dataset,video_id,clip_id,clase_binaria,p_ocultamiento,pred_threshold_final
0,DCSASS,Shoplifting006_x264,Shoplifting006_x264_19,1,0.12620,0
1,DCSASS,Shoplifting006_x264,Shoplifting006_x264_28,1,0.03738,0
2,DCSASS,Shoplifting006_x264,Shoplifting006_x264_29,1,0.01015,0
3,DCSASS,Shoplifting021_x264,Shoplifting021_x264_23,1,0.00945,0
4,MNNIT,shoplifting-13,MNNIT_013,1,0.37100,0
5,MNNIT,shoplifting-65,MNNIT_064,1,0.38400,0
6,MNNIT,shoplifting-66,MNNIT_065,1,0.16090,0
7,MNNIT,shoplifting-67,MNNIT_066,1,0.43920,0
8,MNNIT,shoplifting-68,MNNIT_067,1,0.33400,0
9,MNNIT,shoplifting-69,MNNIT_068,1,0.61570,0


In [5]:
# ============================================================
# Construcción del conjunto de prueba v2
# ============================================================

df_test_v2 = (
    df_test_v1[
        ~df_test_v1["clip_id"].isin(
            AMBIGUOUS_TEST_CLIPS_AVANCE3
        )
    ]
    .copy()
    .reset_index(drop=True)
)

n_v1 = len(df_test_v1)
n_v2 = len(df_test_v2)

positivos_v1 = int(
    df_test_v1["clase_binaria"].sum()
)

positivos_v2 = int(
    df_test_v2["clase_binaria"].sum()
)

negativos_v1 = n_v1 - positivos_v1
negativos_v2 = n_v2 - positivos_v2

ambiguos_restantes = (
    df_test_v2["clip_id"]
    .isin(AMBIGUOUS_TEST_CLIPS_AVANCE3)
    .sum()
)

print("=== CONSTRUCCIÓN DE TEST V2 ===")
print("Registros test v1:", n_v1)
print("Registros test v2:", n_v2)
print("Clips excluidos:", n_v1 - n_v2)
print()
print("Positivos test v1:", positivos_v1)
print("Positivos test v2:", positivos_v2)
print()
print("Negativos test v1:", negativos_v1)
print("Negativos test v2:", negativos_v2)
print()
print(
    "Clips ambiguos restantes en test v2:",
    int(ambiguos_restantes)
)

assert n_v1 == 147
assert n_v2 == 136
assert n_v1 - n_v2 == 11

assert positivos_v1 == 28
assert positivos_v2 == 17

assert negativos_v1 == 119
assert negativos_v2 == 119

assert ambiguos_restantes == 0

=== CONSTRUCCIÓN DE TEST V2 ===
Registros test v1: 147
Registros test v2: 136
Clips excluidos: 11

Positivos test v1: 28
Positivos test v2: 17

Negativos test v1: 119
Negativos test v2: 119

Clips ambiguos restantes en test v2: 0


In [6]:
# ============================================================
# Experimento 0
# Comparación de métricas entre test v1 y test v2
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_curve
)


THRESHOLD_FINAL = 0.7090


def calcular_metricas(
    y_true,
    y_prob,
    threshold=0.5
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    average_precision = average_precision_score(
        y_true,
        y_prob
    )

    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_true,
            y_prob
        )
    )

    pr_auc = auc(
        recall_curve,
        precision_curve
    )

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )

    fnr = 1 - tpr

    idx_eer = np.nanargmin(
        np.abs(fpr - fnr)
    )

    eer = (
        fpr[idx_eer]
        + fnr[idx_eer]
    ) / 2

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc,
        "average_precision": average_precision,
        "pr_auc": pr_auc,
        "eer": eer,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


metricas_v1 = calcular_metricas(
    df_test_v1["clase_binaria"],
    df_test_v1["p_ocultamiento"],
    threshold=THRESHOLD_FINAL
)

metricas_v2 = calcular_metricas(
    df_test_v2["clase_binaria"],
    df_test_v2["p_ocultamiento"],
    threshold=THRESHOLD_FINAL
)


metricas_mostrar = [
    "average_precision",
    "pr_auc",
    "roc_auc",
    "eer",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "tn",
    "fp",
    "fn",
    "tp"
]


comparacion_exp0 = pd.DataFrame(
    {
        "test_v1": [
            metricas_v1[m]
            for m in metricas_mostrar
        ],
        "test_v2": [
            metricas_v2[m]
            for m in metricas_mostrar
        ]
    },
    index=metricas_mostrar
)

comparacion_exp0["delta"] = (
    comparacion_exp0["test_v2"]
    - comparacion_exp0["test_v1"]
)

display(
    comparacion_exp0.round(4)
)

,test_v1,test_v2,delta
average_precision,0.5393,0.6067,0.0675
pr_auc,0.5244,0.5858,0.0615
roc_auc,0.8472,0.9249,0.0776
eer,0.2164,0.1092,-0.1071
accuracy,0.8367,0.9044,0.0677
precision,0.5769,0.5769,0.0000
recall,0.5357,0.8824,0.3466
specificity,0.9076,0.9076,0.0000
f1,0.5556,0.6977,0.1421
tn,108.0000,108.0000,0.0000


### 8.1.1 Resultados del Experimento 0

La evaluación con el ground truth original reproduce las métricas obtenidas en el Avance 3, lo que confirma que las predicciones congeladas del baseline y el procedimiento de cálculo se conservaron correctamente.

Después de excluir los 11 clips previamente clasificados como ambiguos para la definición operacional de `ocultamiento`, el conjunto de prueba pasa de 147 a 136 observaciones. Los 119 ejemplos negativos permanecen sin cambios, mientras que los positivos disminuyen de 28 a 17.

La modificación del ground truth produce cambios relevantes en las métricas del mismo checkpoint R3D-18. Average Precision aumenta de 0.5393 a 0.6067, PR-AUC de 0.5244 a 0.5858 y ROC-AUC de 0.8472 a 0.9249. EER disminuye de 0.2164 a 0.1092.

En las métricas dependientes del umbral seleccionado previamente en validación, recall aumenta de 0.5357 a 0.8824 y F1-score de 0.5556 a 0.6977. Precision permanece en 0.5769 y specificity en 0.9076.

Este comportamiento se explica por la naturaleza de los casos excluidos. Los 11 clips eran positivos que el baseline había clasificado como negativos. Al retirarlos de la evaluación, los falsos negativos disminuyen de 13 a 2, mientras que TP, FP y TN permanecen sin cambios. Por esta razón, recall aumenta de manera considerable, mientras que precision y specificity permanecen iguales.

Los resultados no representan una mejora del modelo. El checkpoint, sus parámetros y sus probabilidades permanecen sin cambios. La diferencia corresponde al efecto de evaluar el mismo modelo con una versión del ground truth más alineada con la definición operacional del proyecto.

Esta revisión tiene un alcance limitado. Los clips excluidos fueron identificados mediante el análisis cualitativo de falsos negativos realizado en el Avance 3 y no mediante una nueva auditoría exhaustiva de todo el conjunto de prueba. Por esta razón, el experimento se interpreta como un análisis de sensibilidad de las métricas ante problemas de alineación semántica de las etiquetas y no como una validación completa de la calidad del ground truth.

A partir de este punto, el conjunto de evaluación revisado queda congelado y no se modificarán sus etiquetas en función de los errores producidos por los modelos alternativos del Avance 4.

In [7]:
# ============================================================
# Congelación del dataset v2 para el Avance 4
# ============================================================

import hashlib


BASE_DIR = (
    REPO_DIR
    / "Datos"
    / "Preparacion_Datasets"
)

SPLIT_V1_PATH = (
    BASE_DIR
    / "baseline_split_equipo41.csv"
)

SPLIT_V2_PATH = (
    BASE_DIR
    / "avance4_split_ground_truth_v2.csv"
)


if not SPLIT_V1_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el archivo esperado: "
        "Datos/Preparacion_Datasets/"
        "baseline_split_equipo41.csv"
    )


df_split_v2 = pd.read_csv(
    SPLIT_V1_PATH
)

assert len(df_split_v2) == 985
assert df_split_v2["clip_id"].notna().all()
assert df_split_v2["split"].notna().all()


mascara_ambiguos = (
    df_split_v2["clip_id"]
    .isin(AMBIGUOUS_TEST_CLIPS_AVANCE3)
)

assert mascara_ambiguos.sum() == 11

assert (
    df_split_v2.loc[
        mascara_ambiguos,
        "split"
    ]
    .eq("test")
    .all()
)

assert (
    df_split_v2.loc[
        mascara_ambiguos,
        "clase_binaria"
    ]
    .eq(1)
    .all()
)


df_split_v2["revision_etiqueta"] = (
    "sin_revision"
)

df_split_v2["elegible_avance4"] = True


df_split_v2.loc[
    mascara_ambiguos,
    "revision_etiqueta"
] = "ambigua_para_ocultamiento"

df_split_v2.loc[
    mascara_ambiguos,
    "elegible_avance4"
] = False


if SPLIT_V2_PATH.exists():

    df_split_v2_existente = pd.read_csv(
        SPLIT_V2_PATH
    )

    pd.testing.assert_frame_equal(
        df_split_v2,
        df_split_v2_existente,
        check_dtype=False
    )

    estado_archivo = (
        "El archivo ya existía y coincide con el dataset v2."
    )

else:

    df_split_v2.to_csv(
        SPLIT_V2_PATH,
        index=False
    )

    estado_archivo = (
        "Dataset v2 guardado correctamente."
    )


def calcular_sha256(ruta):
    sha256 = hashlib.sha256()

    with open(ruta, "rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(8192),
            b""
        ):
            sha256.update(bloque)

    return sha256.hexdigest()


sha256_v2 = calcular_sha256(
    SPLIT_V2_PATH
)


resumen_elegibles = (
    df_split_v2[
        df_split_v2["elegible_avance4"]
    ]
    ["split"]
    .value_counts()
    .reindex(
        ["train", "val", "test"]
    )
)


print("=== DATASET V2 CONGELADO ===")
print(estado_archivo)
print(
    "Archivo:",
    SPLIT_V2_PATH.relative_to(REPO_DIR)
)
print(
    "Registros totales:",
    len(df_split_v2)
)
print(
    "Casos ambiguos excluidos:",
    int((~df_split_v2["elegible_avance4"]).sum())
)

print("\nRegistros elegibles por partición:")
print(resumen_elegibles)

print("\nSHA-256:")
print(sha256_v2)

=== DATASET V2 CONGELADO ===
Dataset v2 guardado correctamente.
Archivo: Datos/Preparacion_Datasets/avance4_split_ground_truth_v2.csv
Registros totales: 985
Casos ambiguos excluidos: 11

Registros elegibles por partición:
split
train    702
val      136
test     136
Name: count, dtype: int64

SHA-256:
b85d6b55e8c32420a4df5a0fb9a10e558105beee47f1d4a98ad83d63fcbda21e


In [8]:
# ============================================================
# Particiones elegibles para los experimentos RGB
# ============================================================

df_modelado_v2 = (
    df_split_v2[
        df_split_v2["elegible_avance4"]
    ]
    .copy()
    .reset_index(drop=True)
)

df_train_v2 = (
    df_modelado_v2[
        df_modelado_v2["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

df_val_v2 = (
    df_modelado_v2[
        df_modelado_v2["split"] == "val"
    ]
    .copy()
    .reset_index(drop=True)
)

df_test_modelado_v2 = (
    df_modelado_v2[
        df_modelado_v2["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


def resumen_particion(nombre, df):
    positivos = int(
        df["clase_binaria"].sum()
    )

    negativos = (
        len(df)
        - positivos
    )

    prevalencia = (
        positivos / len(df)
    )

    return {
        "particion": nombre,
        "n": len(df),
        "negativos": negativos,
        "positivos": positivos,
        "prevalencia_ocultamiento": prevalencia
    }


resumen_particiones_v2 = pd.DataFrame(
    [
        resumen_particion(
            "train",
            df_train_v2
        ),
        resumen_particion(
            "val",
            df_val_v2
        ),
        resumen_particion(
            "test",
            df_test_modelado_v2
        )
    ]
)

assert len(df_train_v2) == 702
assert len(df_val_v2) == 136
assert len(df_test_modelado_v2) == 136

assert (
    df_modelado_v2["revision_etiqueta"]
    .ne("ambigua_para_ocultamiento")
    .all()
)

print("=== PARTICIONES DE MODELADO V2 ===")

display(
    resumen_particiones_v2.style.format(
        {
            "prevalencia_ocultamiento": "{:.4f}"
        }
    )
)

=== PARTICIONES DE MODELADO V2 ===


,particion,n,negativos,positivos,prevalencia_ocultamiento
0,train,702,570,132,0.1880
1,val,136,106,30,0.2206
2,test,136,119,17,0.1250


### 8.1.2 Distribución final del dataset v2

Después de excluir los 11 casos ambiguos identificados en el análisis de falsos negativos del Avance 3, las particiones utilizadas para los experimentos RGB quedan definidas de la siguiente forma:

| Partición | Total | Sin ocultamiento | Ocultamiento | Prevalencia de ocultamiento |
|---|---:|---:|---:|---:|
| Train | 702 | 570 | 132 | 0.1880 |
| Validation | 136 | 106 | 30 | 0.2206 |
| Test | 136 | 119 | 17 | 0.1250 |

Las particiones de entrenamiento y validación permanecen sin cambios respecto al Avance 3. La modificación afecta únicamente al conjunto de prueba, donde los 11 clips excluidos pertenecían a la clase positiva.

La menor prevalencia de `ocultamiento` en test debe considerarse al interpretar las métricas basadas en Precision-Recall. En particular, la referencia esperada para Average Precision de un clasificador no informativo corresponde aproximadamente a la prevalencia de la clase positiva. Para `test_v2` esta referencia es 0.1250.

La diferencia de prevalencia entre las particiones también refuerza la necesidad de utilizar métricas apropiadas para clases desbalanceadas y de evitar que accuracy sea el criterio principal de selección.

A partir de esta sección, todos los modelos RGB utilizarán exactamente estas particiones. Train y validation se emplearán para entrenamiento, selección de checkpoints y ajuste de hiperparámetros. Test se utilizará únicamente después de haber fijado la configuración correspondiente.